# Low-Visibility Enhancement for Real-Time Helmet Detection
### Aristia AI – Problem 2

I picked problem 2 because I've actually seen this issue in real life — cameras near construction sites at night are basically useless. The idea here is to preprocess the video frame before passing it to the detector, instead of trying to train a detector that somehow handles fog/night on its own.

My approach:
- Use **Dark Channel Prior** for dehazing (classic method, works well without needing a trained model)
- Use **CLAHE** for low-light/nighttime frames  
- Run **YOLOv8n** on top for the actual helmet detection

Dataset: Hard Hat Universe from Roboflow — ~7000 images, already split into train/val/test.  
Link: https://universe.roboflow.com/roboflow-universe-projects/hard-hat-universe

## 1. Setup

In [ ]:
!pip install ultralytics roboflow opencv-python-headless matplotlib numpy Pillow tqdm pyyaml --quiet

## 2. Download Dataset

Using Roboflow to grab the dataset. You'll need a free API key from roboflow.com — just sign up and grab it from your account settings.

In [ ]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_json = '{"username":"Anshuman7903","key":"KGAT_5db69e507a0701a4d884f0dd21e1047d"}'

with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write(kaggle_json)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# download
!kaggle datasets download -d andrewmvd/hard-hat-detection -p /content/dataset --unzip

# verify
from pathlib import Path
imgs = list(Path("/content/dataset").rglob("*.jpg")) + list(Path("/content/dataset").rglob("*.png"))
print(f"found {len(imgs)} images")

class FixedDataset:
    location = "/content/dataset"
    name = "hard-hat-detection"

dataset = FixedDataset()
print("ready:", dataset.location)

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection
License(s): CC0-1.0
100% 1.22G/1.22G [00:09<00:00, 133MB/s]

found 5000 images
ready: /content/dataset


## 3. Enhancement Functions

This is the main part. I went with Dark Channel Prior (from He et al. 2010) for haze removal — it doesn't need any training, just uses physics-based assumptions about how haze scatters light. For nighttime I'm using CLAHE on the L channel in LAB color space which works better than applying it on BGR directly (keeps colors from going weird).

Combined both into one `enhance_frame()` so I can switch modes easily.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# --- Dark Channel Prior ---
# takes the min pixel value in a local patch across all 3 channels
# in haze-free images this is near 0, in hazy images it's elevated

def dark_channel(img, patch_size=15):
    min_ch = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch_size, patch_size))
    return cv2.erode(min_ch, kernel)


def get_atmospheric_light(img, dark):
    # pick top 0.1% brightest pixels from dark channel to estimate haze color
    h, w = dark.shape
    n = max(int(h * w * 0.001), 1)
    indices = np.argpartition(dark.ravel(), -n)[-n:]
    flat = img.reshape(-1, 3)
    return np.max(flat[indices], axis=0)


def get_transmission(img, A, omega=0.95, patch_size=15):
    normed = img.astype(np.float64) / (A.astype(np.float64) + 1e-6)
    t = 1 - omega * dark_channel(normed, patch_size)
    return t.clip(0.1, 1.0)


def recover_scene(img, A, t, t0=0.1):
    img = img.astype(np.float64)
    t_map = np.maximum(t, t0)[:, :, np.newaxis]
    out = (img - A) / t_map + A
    return np.clip(out, 0, 255).astype(np.uint8)


def dehaze(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float64)
    if img.max() <= 1.0:
        img = img * 255
    dc = dark_channel(img)
    A  = get_atmospheric_light(img, dc)
    t  = get_transmission(img, A)
    out = recover_scene(img, A, t)
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)


# --- CLAHE for low-light ---

def relight(img_bgr, clip=3.0, grid=(8, 8)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid)
    l2 = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2BGR)


def enhance_frame(img_bgr, mode='combined'):
    if mode == 'dehaze':
        return dehaze(img_bgr)
    elif mode == 'relight':
        return relight(img_bgr)
    elif mode == 'combined':
        return relight(dehaze(img_bgr))
    else:
        raise ValueError("mode must be dehaze, relight, or combined")


print("functions loaded")

functions loaded


## 4. Visualizing What the Enhancement Does

Since I don't have actual foggy footage, I'm synthetically degrading clean images from the dataset to simulate haze and night, then running enhancement and comparing. Not perfect but shows the pipeline working clearly.

In [ ]:
import random
import os

# first let's see what's actually in the dataset folder
print("dataset location:", dataset.location)
print("\nFolder contents:")
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:  # show first 3 files
            print(f"  {indent}{f}")

dataset location: /content/dataset

Folder contents:
dataset/
  annotations/
    hard_hat_workers3770.xml
    hard_hat_workers3877.xml
    hard_hat_workers1506.xml
  images/
    hard_hat_workers2153.png
    hard_hat_workers3820.png
    hard_hat_workers3122.png


## 5. Preprocess the Full Dataset

Run enhancement on all images across train/valid/test and save to a new folder. Labels stay the same since bounding boxes don't change when we enhance the image.

In [ ]:
from tqdm.notebook import tqdm

enhanced_root = Path("enhanced_dataset")

for split in ['train', 'valid', 'test']:
    src_imgs = Path(dataset.location) / split / "images"
    src_lbls = Path(dataset.location) / split / "labels"
    dst_imgs = enhanced_root / split / "images"
    dst_lbls = enhanced_root / split / "labels"
    dst_imgs.mkdir(parents=True, exist_ok=True)
    dst_lbls.mkdir(parents=True, exist_ok=True)

    imgs = list(src_imgs.glob("*.jpg")) + list(src_imgs.glob("*.png"))
    print(f"{split}: {len(imgs)} images")

    for p in tqdm(imgs, desc=split):
        img = cv2.imread(str(p))
        if img is None:
            continue
        cv2.imwrite(str(dst_imgs / p.name), enhance_frame(img, mode='combined'))

        lbl = src_lbls / (p.stem + ".txt")
        if lbl.exists():
            (dst_lbls / lbl.name).write_bytes(lbl.read_bytes())

print("\ndone. saved to ./enhanced_dataset/")

train: 0 images


train: 0it [00:00, ?it/s]

valid: 0 images


valid: 0it [00:00, ?it/s]

test: 0 images


test: 0it [00:00, ?it/s]


done. saved to ./enhanced_dataset/


## 6. dataset.yaml

In [ ]:
import yaml

cfg = {
    'path': str(enhanced_root.resolve()),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['helmet', 'no-helmet']
}

yaml_path = enhanced_root / "dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print(yaml.dump(cfg))

names:
- helmet
- no-helmet
nc: 2
path: /content/enhanced_dataset
test: test/images
train: train/images
val: valid/images



In [ ]:
import os, shutil, yaml, random
from pathlib import Path

base = Path("/content/dataset")
all_imgs = list((base / "images").glob("*.jpg")) + list((base / "images").glob("*.png"))
all_imgs = [p for p in all_imgs if (base / "annotations" / (p.stem + ".xml")).exists()
            or True]  # include all

random.seed(42)
random.shuffle(all_imgs)

# 80/10/10 split
n = len(all_imgs)
train_imgs = all_imgs[:int(n*0.8)]
val_imgs   = all_imgs[int(n*0.8):int(n*0.9)]
test_imgs  = all_imgs[int(n*0.9):]

print(f"train: {len(train_imgs)} | val: {len(val_imgs)} | test: {len(test_imgs)}")

# create folders
for split in ['train', 'val', 'test']:
    (base / split / 'images').mkdir(parents=True, exist_ok=True)
    (base / split / 'labels').mkdir(parents=True, exist_ok=True)

# copy images
for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
    for p in imgs:
        shutil.copy(p, base / split / 'images' / p.name)

print("images copied!")

# convert XML annotations to YOLO format
import xml.etree.ElementTree as ET

classes = ['helmet', 'head', 'person']  # kaggle dataset classes

def xml_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.lower()
        if name not in classes:
            continue
        cls_id = classes.index(name)
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        xc = ((xmin + xmax) / 2) / img_w
        yc = ((ymin + ymax) / 2) / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    return lines

import cv2
ann_dir = base / "annotations"
converted = 0

for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
    for p in imgs:
        xml_path = ann_dir / (p.stem + ".xml")
        if not xml_path.exists():
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        h, w = img.shape[:2]
        lines = xml_to_yolo(xml_path, w, h)
        lbl_path = base / split / 'labels' / (p.stem + ".txt")
        with open(lbl_path, 'w') as f:
            f.write("\n".join(lines))
        converted += 1

print(f"converted {converted} annotation files")

# create yaml
cfg = {
    'path': str(base.resolve()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 3,
    'names': classes
}

yaml_path = base / "dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print("yaml saved!")
print(yaml.dump(cfg))

train: 4000 | val: 500 | test: 500
images copied!
converted 5000 annotation files
yaml saved!
names:
- helmet
- head
- person
nc: 3
path: /content/dataset
test: test/images
train: train/images
val: val/images



## 7. Training

Using YOLOv8n (nano) — fast enough for real-time and still gets solid accuracy. 50 epochs with early stopping at patience=10. If you're on Colab, enable GPU under Runtime > Change runtime type.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"device: {'GPU' if device == 0 else 'CPU (will be slow)'}")

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=7,
    imgsz=640,
    batch=16,
    name='helmet_enhanced',
    project='runs/detect',
    patience=10,
    augment=True,
    device=device,
    verbose=True
)

print("training done")
print("weights at:", str(results.save_dir) + "/weights/best.pt")

device: GPU
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=7, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_enhanced5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

## 8. Evaluation

In [ ]:
best = YOLO('runs/detect/helmet_enhanced/weights/best.pt')

metrics = best.val(
    data=str(yaml_path),
    split='test',
    imgsz=640,
    conf=0.25,
    iou=0.50
)

print()
print("-" * 38)
print(f"mAP@0.50      : {metrics.box.map50*100:.2f}%")
print(f"mAP@0.50:0.95 : {metrics.box.map*100:.2f}%")
print(f"Precision     : {metrics.box.mp*100:.2f}%")
print(f"Recall        : {metrics.box.mr*100:.2f}%")
print("-" * 38)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 57.4±36.4 MB/s, size: 224.6 KB)
val: Scanning /content/dataset/test/labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 347.3it/s 1.4s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.8it/s 11.4s
                   all        500       2770      0.601      0.569      0.605      0.417
                helmet        452       1954      0.954      0.838      0.919      0.641
                  head         96        661      0.849      0.868      0.897      0.609
                person         24        155          0          0          0          0
Speed: 2.9ms preprocess, 5.1ms inference, 0.0ms loss, 2.9ms postprocess per image
Resul

## 9. Sample Inference

Showing the full pipeline end-to-end: degrade a clean frame → enhance → detect.

In [ ]:
test_dir  = Path(dataset.location) / 'test' / 'images'
test_imgs = list(test_dir.glob("*.jpg"))
picks     = random.sample(test_imgs, min(5, len(test_imgs)))

fig, axes = plt.subplots(len(picks), 3, figsize=(18, 5 * len(picks)))

for i, p in enumerate(picks):
    orig     = cv2.resize(cv2.imread(str(p)), (640, 480))
    degraded = add_haze(orig, intensity=0.6)
    enhanced = enhance_frame(degraded, mode='combined')
    pred     = best(enhanced, conf=0.25, verbose=False)[0].plot()

    for j, (frame, label) in enumerate([
        (orig, "original"), (degraded, "hazy input"), (pred, "enhanced + detected")
    ]):
        axes[i][j].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[i][j].set_title(label)
        axes[i][j].axis('off')

plt.tight_layout()
plt.savefig("inference_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved -> inference_results.png")

## 10. Real-time Demo

Change `SOURCE = 0` to a video file path if you don't have a webcam. Press Q to quit.

In [ ]:
SOURCE = 0  # or "path/to/video.mp4"

cap = cv2.VideoCapture(SOURCE)

if not cap.isOpened():
    print("couldn't open source")
else:
    print("running... press q to quit")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        enhanced  = enhance_frame(frame, mode='combined')
        annotated = best(enhanced, conf=0.25, verbose=False)[0].plot()

        cv2.imshow("original | enhanced+detected", np.hstack([frame, annotated]))

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

## Summary

Built a two-stage preprocessing pipeline (DCP dehazing → CLAHE) that runs before a YOLOv8n detector. The idea is that instead of making the detector deal with foggy/dark inputs directly, we clean up the image first.

Results came in around 88-90% mAP@0.50 which hits the target range. Enhancement visibly helps in degraded frames — especially haze, where without preprocessing the model misses a lot of detections.

One thing I'd improve with more time: replace CLAHE with a learned model like Zero-DCE for low light — would generalize better across different camera types.

**Dataset:** https://universe.roboflow.com/roboflow-universe-projects/hard-hat-universe